In [ ]:
import pickle as pkl
import json
import pandas as pd
from tqdm.auto import tqdm

In [ ]:
with open('software_relations_per_article.pkl', 'rb') as f:
    software_relations=pkl.load(f)

In [ ]:
software_relations[0:10]

In [ ]:
with open('hardware_relations_per_article.pkl', 'rb') as f:
    hardware_relations=pkl.load(f)

In [ ]:
hardware_relations[0:100]

In [ ]:
with open('cve.json', 'r') as f:
    cve_list=json.load(f)

In [ ]:
cve_list[2]

In [ ]:
cve_list[0]['identifier']

In [ ]:
#ONLY CVE Vulnerabilities:

In [ ]:
import re
from typing import Iterable, List, Dict, Any, Set, Tuple

# Match CVE with normal or Unicode dashes, any spacing, case-insensitive.
_CVE_RE = re.compile(
    r"\bCVE[\-\u2010-\u2015\u2212 ]?(\d{4})[\-\u2010-\u2015\u2212 ]?(\d+)\b",
    re.IGNORECASE,
)

def _normalize_cve_id(year: str, num: str) -> str:
    # CVE canonical form (no extra zero-padding assumptions).
    return f"CVE-{year}-{num}".upper()

def find_all_cves(text: str) -> List[str]:
    """Return all normalized CVE IDs found in text."""
    return [_normalize_cve_id(m.group(1), m.group(2)) for m in _CVE_RE.finditer(text)]

def build_cve_set(cve_records: Iterable[Dict[str, Any]]) -> Set[str]:
    """Build a normalized set of known CVE IDs from records with 'identifier'."""
    s: Set[str] = set()
    for rec in cve_records:
        ident = rec.get("identifier", "")
        for cve in find_all_cves(ident):
            s.add(cve)
    return s

def filter_to_all_cve_entries_and_collect_missing(
    list_of_lists: List[List[str]],
    known_cves: Set[str],
) -> Tuple[List[List[str]], Set[str]]:
    """
    Returns:
      - filtered list-of-lists (same shape), where each sublist only contains items
        whose 'vulnerability' (after the first comma) contains at least one CVE.
        (No check against known_cves for inclusion in the output.)
      - a set of CVE IDs that were seen in the input but NOT present in known_cves.
    """
    filtered: List[List[str]] = []
    missing: Set[str] = set()

    for sub in list_of_lists:
        if not sub:
            filtered.append([])  # preserve empty sublist
            continue

        kept = []
        for item in sub:
            # Split into "Software/Hardware", "Vulnerability"
            _, vuln = (item.split(",", 1) + [""])[:2]
            cves = find_all_cves(vuln)

            if cves:
                kept.append(item)  # keep original string unchanged
                # Track which CVEs are not in the known dataset
                for c in cves:
                    if c not in known_cves:
                        missing.add(c)

        filtered.append(kept)

    return filtered, missing

# ---------------------------
# Example usage
# ---------------------------

# data = [
#     [],
#     [],
#     [],
#     ['American Express, phishing-as-a-service (PhaaS)',
#      'Bank of America, phishing-as-a-service (PhaaS)',
#      'DHL, phishing-as-a-service (PhaaS)',
#      'Microsoft, phishing-as-a-service (PhaaS)',
#      'Naver, phishing-as-a-service (PhaaS)',
#      'American Express, BulletProofLink',
#      'Bank of America, BulletProofLink',
#      'DHL, BulletProofLink',
#      'Microsoft, BulletProofLink',
#      'Naver, BulletProofLink',
#      'American Express, BulletProftLink',
#      'Bank of America, BulletProftLink',
#      'DHL, BulletProftLink',
#      'Microsoft, BulletProftLink',
#      'Naver, BulletProftLink',
#      'DRACOON, adversary-in-the-middle (AiTM) attacks'],
#     ['Windows, BiBi-Windows Wiper',
#      'Linux, BiBi-Linux Wiper',
#      'Windows, bibi.exe'],
#     ['macOS, ObjCShellz', 'macOS, RustBucket'],
#     [],
#     ['MicroSCADA supervisory control system, living-off-the-land (LotL)'],
#     [],
#     ['Atlassian Confluence Data Center and Server, CVE-2023-22515',
#      'Confluence, CVE-2023-22515',
#      'Confluence servers, CVE-2023-22515',
#      'Atlassian Confluence Data Center and Server, CVE-2023-22518',
#      'Confluence servers, CVE-2023-22518',
#      'Confluence, CVE-2023-22518',
#      'Confluence, web shell',
#      'Confluence servers, web shell',
#      'JIRA, web shell',
#      'Bitbucket, web shell']
# ]

# cve_records = [
#     {'identifier': 'CVE-2006-3783', 'description': {'description_data': [{'lang': 'en','value': '...'}]}},
#     # In reality you'll have ~216k CVEs here
# ]

known_cves = build_cve_set(cve_list)

filtered_list_software, missing_cves_software = filter_to_all_cve_entries_and_collect_missing(software_relations, known_cves)
filtered_list_hardware, missing_cves_hardware = filter_to_all_cve_entries_and_collect_missing(hardware_relations, known_cves)

# print(filtered_list)  # same outer shape; only entries with a CVE after the comma remain
# print(sorted(missing_cves)[:10])  # sample of CVEs that appeared in the data but aren't in known_cves


# known_cves = build_cve_set(cve_list)
# only_cves_software = filter_to_known_cves(software_relations, known_cves)
# only_cves_hardware = filter_to_known_cves(hardware_relations, known_cves)


# 'result' preserves the outer shape; only items whose post-comma part
# has a CVE present in known_cves remain. With the single CVE above,
# all CVE-2023-... entries will be dropped; sublists with no matches stay [].
# print(result)


In [ ]:
filtered_list_software[0:100]

In [ ]:
filtered_list_hardware[0:200]

In [ ]:
missing_cves_software

In [ ]:
missing_cves_hardware

In [ ]:
cisa_dataset=pd.read_csv('known_exploited_vulnerabilities.csv')

In [ ]:
cisa_dataset.head()

In [ ]:
import re
import pandas as pd
from typing import List, Dict, Any, Tuple, Set
try:
    from rapidfuzz import fuzz
    HAVE_RAPIDFUZZ = True
    print("True")
except Exception:
    HAVE_RAPIDFUZZ = False

# ----------------------
# Helpers: parsing & normalization
# ----------------------

_CVE_RE = re.compile(r"\bCVE[\-\u2010-\u2015\u2212 ]?(\d{4})[\-\u2010-\u2015\u2212 ]?(\d+)\b", re.IGNORECASE)

def extract_pair(entry: str) -> Tuple[str | None, str | None]:
    """
    From 'Software/Hardware, CVE-YYYY-NNNN ...' return (software, CVE-ID).
    Returns (None, None) if not parseable.
    """
    parts = entry.split(",", 1)
    if not parts:
        return (None, None)
    sw = parts[0].strip()
    cve = None
    if len(parts) > 1:
        m = _CVE_RE.search(parts[1])
        if m:
            cve = f"CVE-{m.group(1)}-{m.group(2)}".upper()
    return (sw if sw else None, cve)

def norm_text(s: str) -> str:
    # Lower, replace non-alnum with spaces, collapse spaces
    s = s.lower()
    s = re.sub(r"[^a-z0-9]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def token_set(s: str) -> Set[str]:
    return set(norm_text(s).split())

def token_subset(a: str, b: str) -> bool:
    """
    Are all tokens in a contained in b? (a ⊆ b)
    Robust against punctuation/casing.
    """
    A = token_set(a)
    B = token_set(b)
    return len(A) > 0 and A.issubset(B)

def fuzzy_match(a: str, b: str, threshold: int = 85) -> bool:
    if not HAVE_RAPIDFUZZ:
        return False
    return fuzz.partial_ratio(a, b) >= threshold

# ----------------------
# Build lookups from sources
# ----------------------

def build_cisa_map(cisa_csv_path: str) -> Dict[str, Set[str]]:
    """
    Map CVE -> candidate software/hardware strings gathered from CISA columns:
    vendorProject, product, and their concatenation.
    """
    df = pd.read_csv(cisa_csv_path)
    df["cveID"] = df["cveID"].astype(str).str.upper().str.strip()
    candidates: Dict[str, Set[str]] = {}
    for _, row in df.iterrows():
        cve = row["cveID"]
        vals = set()
        for col in ("vendorProject", "product"):
            if pd.notna(row.get(col, None)):
                vals.add(str(row[col]))
        # Also add "Vendor Product" combined, common for human-written names
        if pd.notna(row.get("vendorProject", None)) and pd.notna(row.get("product", None)):
            vals.add(f"{row['vendorProject']} {row['product']}")
        if cve not in candidates:
            candidates[cve] = set()
        candidates[cve].update(v for v in vals if v and v.strip())
    return candidates

def build_cve_desc_map(cve_records: List[Dict[str, Any]]) -> Dict[str, Dict[str, Any]]:
    """
    Map CVE -> dict with:
      - 'desc': concatenated English descriptions (string)
      - 'urls': list of URLs (strings)
    """
    out: Dict[str, Dict[str, Any]] = {}
    for rec in cve_records:
        ident = rec.get("identifier", "")
        m = _CVE_RE.search(ident)
        if not m:
            continue
        cve_id = f"CVE-{m.group(1)}-{m.group(2)}".upper()

        # gather english descriptions
        desc = []
        desc_data = rec.get("description", {}).get("description_data", [])
        for d in desc_data:
            if d.get("lang", "").lower() == "en":
                desc.append(str(d.get("value", "")))
        desc_text = " ".join(desc)

        urls = [str(u) for u in rec.get("url", []) if isinstance(u, str)]

        out[cve_id] = {
            "desc": desc_text,
            "urls": urls,
        }
    return out

# ----------------------
# Pair presence checks
# ----------------------

def pair_present_in_cisa(sw: str, cve: str, cisa_map: Dict[str, Set[str]]) -> bool:
    if cve not in cisa_map:
        return False
    sw_norm = norm_text(sw)
    for cand in cisa_map[cve]:
        if token_subset(sw, cand):
            return True
        if token_subset(cand, sw):
            return True
        if HAVE_RAPIDFUZZ and fuzzy_match(sw, cand, threshold=88):
            return True
    return False

def pair_present_in_cve_json(sw: str, cve: str, cve_desc_map: Dict[str, Dict[str, Any]]) -> bool:
    info = cve_desc_map.get(cve)
    if not info:
        return False
    sw_norm = norm_text(sw)
    # 1) Token-subset on description
    if token_subset(sw, info.get("desc", "")):
        return True
    # 2) Token-subset on any URL (often reveals product/vendor)
    for u in info.get("urls", []):
        if token_subset(sw, u):
            return True
    # 3) Optional fuzzy on description for robustness
    if HAVE_RAPIDFUZZ and fuzzy_match(sw, info.get("desc", ""), threshold=90):
        return True
    return False

def classify_pairs_preserve_shape(
    list_of_lists: List[List[str]],
    cisa_map: Dict[str, Set[str]],
    cve_desc_map: Dict[str, Dict[str, Any]],
) -> Tuple[List[List[str]], List[List[str]]]:
    """
    Returns (type_a_present, type_b_missing), each a list-of-lists with the SAME shape
    as input. Each entry is the original string.
    """
    type_a: List[List[str]] = []
    type_b: List[List[str]] = []

    for sub in tqdm(list_of_lists):
        if not sub:
            type_a.append([])
            type_b.append([])
            continue

        a_bucket, b_bucket = [], []
        for entry in sub:
            sw, cve = extract_pair(entry)
            if not sw or not cve:
                # Not a (software, CVE) pair → it doesn't belong in either bucket
                continue

            present = pair_present_in_cisa(sw, cve, cisa_map) or pair_present_in_cve_json(sw, cve, cve_desc_map)

            if present:
                a_bucket.append(entry)  # Type A
            else:
                b_bucket.append(entry)  # Type B

        type_a.append(a_bucket)
        type_b.append(b_bucket)

    return type_a, type_b


In [ ]:
cisa_map = build_cisa_map("known_exploited_vulnerabilities.csv")

In [ ]:
cve_desc_map = build_cve_desc_map(cve_list)

In [ ]:
cve_desc_map["CVE-2023-22515"]["desc"]


In [ ]:
type_a_software, type_b_software = classify_pairs_preserve_shape(filtered_list_software, cisa_map, cve_desc_map)
type_a_hardware, type_b_hardware = classify_pairs_preserve_shape(filtered_list_hardware, cisa_map, cve_desc_map)


In [ ]:
cve_list[12344]

In [ ]:
cisa_dataset.head()

In [ ]:
#For Type C

In [ ]:
# ============================================================
# FAST SHAPE-PRESERVING NATURAL-LANGUAGE → CVE MATCHING PIPELINE
# ============================================================
# - Precomputes normalized corpora + inverted index
# - Generates candidates by token overlap (cheap)
# - Prunes to top-K before fuzzy checks (fast)
# - Preserves original list-of-lists shape
#
# Requires: pandas, (optional) rapidfuzz
# ============================================================

import re
import json
import math
import pandas as pd
from collections import defaultdict, Counter

try:
    from rapidfuzz import fuzz
    HAVE_RAPIDFUZZ = True
except Exception:
    HAVE_RAPIDFUZZ = False

# -----------------------------
# CONFIG (tune for speed/quality)
# -----------------------------
MIN_TOKEN_LEN = 4          # ignore tokens shorter than this when indexing
MAX_CANDIDATES = 80        # how many candidates to keep after token-overlap scoring
FUZZY_THRESHOLD = 85       # partial_ratio threshold (if rapidfuzz is available)

# -----------------------------
# REGEX (precompiled)
# -----------------------------
_CVE_RE = re.compile(
    r"\bCVE[\-\u2010-\u2015\u2212 ]?(\d{4})[\-\u2010-\u2015\u2212 ]?(\d+)\b",
    re.IGNORECASE
)
_PAT_NON_ALNUM = re.compile(r"[^a-z0-9]+", re.IGNORECASE)
_PAT_SPACES = re.compile(r"\s+")

STOP_TOKENS = {
    # very generic words you might see everywhere; keeps index small
    "the","and","with","for","from","into","over","under","that","this","those","these",
    "vulnerability","vulnerabilities","issue","issues","plugin","plugins","product","products",
    "allows","could","may","might","lead","leading","contain","contains","containing","via",
    "using","used","improper","insufficient","validation","check","checks","bypass","dos","denial",
    "service","remote","code","execution","arbitrary","information","disclosure","local","attack",
    "attacker","attackers","crafted","malformed","request","requests","response","responses",
    "unauthenticated","authenticated","privilege","privileges","elevation","escalation","xss","sql",
    "injection","rce","lotl","aitm","phishing","paas","service","services","api"
}

# -----------------------------
# TEXT NORMALIZATION
# -----------------------------
def norm_text(s: str) -> str:
    if not isinstance(s, str):
        s = "" if s is None else str(s)
    s = s.lower()
    s = _PAT_NON_ALNUM.sub(" ", s)
    s = _PAT_SPACES.sub(" ", s).strip()
    return s

def tokenize(s: str):
    return [t for t in norm_text(s).split() if len(t) >= MIN_TOKEN_LEN and t not in STOP_TOKENS]

def extract_pair(entry):
    """
    Return (software, cve_id_or_None, vuln_text).
    vuln_text = text after the first comma if present, else the whole entry.
    """
    parts = entry.split(",", 1)
    sw = parts[0].strip()
    vuln = parts[1].strip() if len(parts) > 1 else entry.strip()
    m = _CVE_RE.search(vuln)
    cve = None
    if m:
        cve = f"CVE-{m.group(1)}-{m.group(2)}".upper()
    return sw, cve, vuln

# -----------------------------
# BUILD CISA TEXT CORPUS + INDEX
# -----------------------------
def build_cisa_text_fields(cisa_df: pd.DataFrame):
    """
    Returns:
      cisa_texts: dict CVE -> normalized searchable text
    """
    cisa_texts = {}
    for _, row in cisa_df.iterrows():
        cve = str(row["cveID"]).strip().upper()
        texts = []
        for col in ("vulnerabilityName","shortDescription","notes","vendorProject","product"):
            if col in row and pd.notna(row[col]):
                texts.append(str(row[col]))
        cisa_texts[cve] = norm_text(" ".join(texts))
    return cisa_texts

# -----------------------------
# BUILD CVE DESCRIPTION CORPUS + INDEX
# -----------------------------
def build_cve_description_text(cve_records):
    """
    Returns:
      cve_texts: dict CVE -> normalized description text
    """
    cve_texts = {}
    for rec in cve_records:
        ident = rec.get("identifier", "")
        m = _CVE_RE.search(ident)
        if not m:
            continue
        cve = f"CVE-{m.group(1)}-{m.group(2)}".upper()

        desc_parts = []
        for d in rec.get("description", {}).get("description_data", []):
            if d.get("lang","").lower() == "en":
                v = d.get("value","")
                if isinstance(v, str):
                    desc_parts.append(v)
        cve_texts[cve] = norm_text(" ".join(desc_parts))
    return cve_texts

# -----------------------------
# BUILD INVERTED INDEX
# -----------------------------
def build_inverted_index(corpus_text_by_cve: dict):
    """
    Input: dict CVE -> normalized text
    Output:
      inv: token -> set(CVE)
      tokens_by_cve: dict CVE -> set(tokens)  (for quick overlap scoring)
    """
    inv = defaultdict(set)
    tokens_by_cve = {}
    for cve, text in tqdm(corpus_text_by_cve.items()):
        toks = set(tokenize(text))
        tokens_by_cve[cve] = toks
        for t in toks:
            inv[t].add(cve)
    return inv, tokens_by_cve

# -----------------------------
# CANDIDATE GENERATION
# -----------------------------
def generate_candidates(nl_text: str, inv_cisa, inv_cve):
    """
    Union of postings lists from both indices for the tokens in nl_text.
    """
    toks = set(tokenize(nl_text))
    if not toks:
        return set()

    cand = set()
    for t in toks:
        if t in inv_cisa:
            cand |= inv_cisa[t]
        if t in inv_cve:
            cand |= inv_cve[t]
    return cand

# -----------------------------
# SCORE & PRUNE CANDIDATES
# -----------------------------
def score_candidates(nl_text: str, candidates: set, cisa_texts: dict, cve_texts: dict,
                     cisa_tokens_by_cve: dict, cve_tokens_by_cve: dict):
    """
    Score by token-overlap count first (cheap), then prune.
    Return list of (cve, score) sorted desc.
    """
    nl_toks = set(tokenize(nl_text))
    scores = []
    for cve in candidates:
        if cve in cisa_tokens_by_cve:
            toks = cisa_tokens_by_cve[cve]
        else:
            toks = cve_tokens_by_cve.get(cve, set())
        overlap = len(nl_toks & toks)
        if overlap:
            scores.append((cve, overlap))
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:MAX_CANDIDATES]

# -----------------------------
# HEAVY CHECK (on pruned set only)
# -----------------------------
def heavy_check(nl_text: str, cve: str, cisa_texts: dict, cve_texts: dict):
    """
    Run substring / token-subset / optional fuzzy on the actual strings.
    """
    nl_norm = norm_text(nl_text)

    # Prefer CISA text if available, else CVE text.
    base_text = cisa_texts.get(cve) or cve_texts.get(cve) or ""
    if not base_text:
        return False

    # substring
    if nl_norm in base_text:
        return True

    # token subset
    nl_set = set(nl_norm.split())
    base_set = set(base_text.split())
    if nl_set and nl_set.issubset(base_set):
        return True

    # fuzzy
    if HAVE_RAPIDFUZZ:
        if fuzz.partial_ratio(nl_norm, base_text) >= FUZZY_THRESHOLD:
            return True

    return False

# -----------------------------
# MATCH SINGLE NATURAL-LANG VULN
# -----------------------------
_MATCH_CACHE = {}

def match_nl_vuln_to_cve_fast(nl_text: str,
                              inv_cisa, inv_cve,
                              cisa_texts, cve_texts,
                              cisa_tokens_by_cve, cve_tokens_by_cve):
    """
    Fast matching:
      1) candidate generation via inverted index
      2) prune by token-overlap
      3) heavy check on top-K
    Uses a small memo cache for repeated strings.
    """
    key = nl_text.strip().lower()
    if key in _MATCH_CACHE:
        return _MATCH_CACHE[key]

    cands = generate_candidates(nl_text, inv_cisa, inv_cve)
    if not cands:
        _MATCH_CACHE[key] = None
        return None

    pruned = score_candidates(nl_text, cands, cisa_texts, cve_texts,
                              cisa_tokens_by_cve, cve_tokens_by_cve)
    for cve, _ in pruned:
        if heavy_check(nl_text, cve, cisa_texts, cve_texts):
            _MATCH_CACHE[key] = cve
            return cve

    _MATCH_CACHE[key] = None
    return None

# -----------------------------
# SHAPE-PRESERVING CLASSIFIER
# -----------------------------
def classify_nl_vulnerabilities_preserve_shape(data,
                                               inv_cisa, inv_cve,
                                               cisa_texts, cve_texts,
                                               cisa_tokens_by_cve, cve_tokens_by_cve):
    """
    Input: list-of-lists of entries.
    Drop CVE entries; for natural-language entries attempt match -> Type C.
    Returns:
      type_c:     same shape, entries as (original_entry, matched_CVE)
      remaining:  same shape, entries as original strings
    """
    type_c = []
    remaining = []

    for sub in tqdm(data):
        sub_type_c = []
        sub_remaining = []

        for entry in sub:
            sw, cve, vuln_text = extract_pair(entry)

            # DROP explicit CVE entries
            if cve:
                continue

            matched = match_nl_vuln_to_cve_fast(
                vuln_text,
                inv_cisa, inv_cve,
                cisa_texts, cve_texts,
                cisa_tokens_by_cve, cve_tokens_by_cve
            )
            if matched:
                sub_type_c.append((entry, matched))
            else:
                sub_remaining.append(entry)

        type_c.append(sub_type_c)
        remaining.append(sub_remaining)

    return type_c, remaining

# -----------------------------
# END-TO-END EXAMPLE USAGE
# -----------------------------
if __name__ == "__main__":
    # Example shapes (replace with your real data variables)
    # software_relations = [...]
    # hardware_relations = [...]

    # Load CISA dataset
    # Ensure the file has columns: cveID, vendorProject, product, vulnerabilityName, shortDescription, notes, ...
    cisa_df = pd.read_csv("known_exploited_vulnerabilities.csv")
    cisa_texts = build_cisa_text_fields(cisa_df)

    # Load CVE JSON dataset (list or jsonl)
    with open("cve.json", "r") as f:
        cve_records = json.load(f)
    cve_texts = build_cve_description_text(cve_records)

    # Merge keys so inverted indices cover both corpora independently
    inv_cisa, cisa_tokens_by_cve = build_inverted_index(cisa_texts)
    inv_cve,  cve_tokens_by_cve  = build_inverted_index(cve_texts)

    # ---- CLASSIFY SOFTWARE RELATIONS ----
    # type_c_software and remaining_software keep the same outer shape as software_relations
    # Example call (uncomment when you have software_relations defined):
    type_c_software, remaining_software = classify_nl_vulnerabilities_preserve_shape(
        software_relations,
        inv_cisa, inv_cve,
        cisa_texts, cve_texts,
        cisa_tokens_by_cve, cve_tokens_by_cve
    )

    # ---- CLASSIFY HARDWARE RELATIONS ----
    # Example call (uncomment when you have hardware_relations defined):
    type_c_hardware, remaining_hardware = classify_nl_vulnerabilities_preserve_shape(
        hardware_relations,
        inv_cisa, inv_cve,
        cisa_texts, cve_texts,
        cisa_tokens_by_cve, cve_tokens_by_cve
    )

    # Example print:
    # print("TYPE C (software):", type_c_software[:2])
    # print("REMAINING (software):", remaining_software[:2])


In [ ]:
print("==== TYPE C MATCHES ====")
print(type_c_software)

print("\n==== REMAINING (Type D/E candidates) ====")
print(remaining_software)

In [ ]:
print("==== TYPE C MATCHES ====")
print(type_c_hardware)

print("\n==== REMAINING (Type D/E candidates) ====")
print(remaining_hardware)

In [ ]:
software_relations[0:10]

In [ ]:
# ============================================================
# TYPE D / E MATCHING — CWE DATASET (ROBUST VERSION)
# ============================================================

import re, json
from rapidfuzz import fuzz
from collections import defaultdict

# -----------------------------
# BASIC NORMALIZATION HELPERS
# -----------------------------
_PAT_NON_ALNUM = re.compile(r"[^a-z0-9]+", re.IGNORECASE)
_PAT_SPACES = re.compile(r"\s+")

def norm_text(s: str) -> str:
    """Normalize to lowercase, alphanumeric, collapse spaces."""
    if not isinstance(s, str):
        s = "" if s is None else str(s)
    s = s.lower()
    s = _PAT_NON_ALNUM.sub(" ", s)
    return _PAT_SPACES.sub(" ", s).strip()

def tokenize(s: str):
    return norm_text(s).split()

def token_subset(a: str, b: str) -> bool:
    A, B = set(tokenize(a)), set(tokenize(b))
    return A and A.issubset(B)

# -----------------------------
# BUILD CWE TEXTS + INVERTED INDEX (SAFE)
# -----------------------------
def build_cwe_texts(cwe_records):
    """
    Build:
      cwe_texts: dict CWE-ID -> normalized joined text
      inv_cwe:   dict token  -> set(CWE-IDs)
    Handles None values gracefully.
    """
    cwe_texts = {}
    inv_cwe = defaultdict(set)

    for rec in cwe_records:
        cwe_id = f"CWE-{rec.get('identifier', '')}".strip()
        if not cwe_id or cwe_id == "CWE-":
            continue

        # Safely extract text fields (replace None with "")
        name = rec.get("name") or ""
        desc = rec.get("description") or ""
        alt  = rec.get("Alternate_Terms") or ""

        joined = " ".join(map(str, [name, desc, alt]))
        text = norm_text(joined)
        cwe_texts[cwe_id] = text

        for tok in set(tokenize(text)):
            if len(tok) >= 4:
                inv_cwe[tok].add(cwe_id)

    return cwe_texts, inv_cwe

# -----------------------------
# MATCHING FUNCTIONS
# -----------------------------
def match_to_cwe(nl_text, cwe_texts, inv_cwe, fuzzy_threshold=85):
    toks = set(tokenize(nl_text))
    if not toks:
        return None

    # candidate generation
    candidates = set()
    for t in toks:
        if t in inv_cwe:
            candidates |= inv_cwe[t]
    if not candidates:
        return None

    # rank by overlap
    scores = []
    for cwe in candidates:
        overlap = len(toks & set(tokenize(cwe_texts[cwe])))
        if overlap:
            scores.append((cwe, overlap))
    scores.sort(key=lambda x: x[1], reverse=True)
    topk = [cwe for cwe, _ in scores[:50]]

    # heavy check on top candidates
    nl_norm = norm_text(nl_text)
    for cwe in topk:
        text = cwe_texts[cwe]
        if nl_norm in text or token_subset(nl_norm, text):
            return cwe
        if fuzz.partial_ratio(nl_norm, text) >= fuzzy_threshold:
            return cwe
    return None

# -----------------------------
# SHAPE-PRESERVING CLASSIFIER
# -----------------------------
def classify_remaining_to_cwe_preserve_shape(remaining_lists, cwe_texts, inv_cwe):
    """
    Input: list-of-lists of remaining vulnerability strings.
    Output:
      type_d: list-of-lists [(entry, matched_CWE)]
      type_e: list-of-lists [entry]
    Shape and order preserved.
    """
    type_d, type_e = [], []

    for sub in remaining_lists:
        sub_d, sub_e = [], []
        for entry in sub:
            vuln_text = entry.split(",", 1)[1].strip() if "," in entry else entry
            match = match_to_cwe(vuln_text, cwe_texts, inv_cwe)
            if match:
                sub_d.append((entry, match))
            else:
                sub_e.append(entry)
        type_d.append(sub_d)
        type_e.append(sub_e)

    return type_d, type_e

# -----------------------------
# EXAMPLE USAGE
# -----------------------------
# Load CWE dataset
with open("cwe.json", "r") as f:
    cwe_records = json.load(f)

cwe_texts, inv_cwe = build_cwe_texts(cwe_records)

# # Example "remaining" list (from previous pipeline)
# remaining_software = [
#     ["Windows, insecure cookie", "Linux, improper input validation"],
#     []
# ]

type_d_software, type_e_software = classify_remaining_to_cwe_preserve_shape(
    remaining_software, cwe_texts, inv_cwe
)

print("TYPE D (matched CWE):")
print(len(type_d_software))
print("\nTYPE E (still unmatched):")
print(len(type_e_software))


In [ ]:
type_d_hardware, type_e_hardware = classify_remaining_to_cwe_preserve_shape(
    remaining_hardware, cwe_texts, inv_cwe
)

print("TYPE D (matched CWE):")
print(len(type_d_hardware))
print("\nTYPE E (still unmatched):")
print(len(type_e_hardware))

In [ ]:
type_d_hardware

In [ ]:
type_e_hardware

In [ ]:
len(type_a_software)
len(type_b_software)
len(type_c_software)
len(type_d_software)
len(type_e_software)

In [ ]:
len(type_a_hardware)
len(type_b_hardware)
len(type_c_hardware)
len(type_d_hardware)
len(type_e_hardware)

In [ ]:
with open('hackerNews_updated.json', 'r') as f:
    hacker_news=json.load(f)

In [ ]:
hacker_news['1241']

In [ ]:
with open("classification_type_results.json", "w") as f:
    json.dump({
        "type_a_software": type_a_software,
        "type_b_software": type_b_software,
        "type_c_software": type_c_software,
        "type_d_software": type_d_software,
        "type_e_software": type_e_software,
        "type_a_hardware": type_a_hardware,
        "type_b_hardware": type_b_hardware,
        "type_c_hardware": type_c_hardware,
        "type_d_hardware": type_d_hardware,
        "type_e_hardware": type_e_hardware,
    }, f, indent=2)

In [ ]:
import json

In [ ]:
with open("classification_type_results.json", "r") as f:
    json_results=json.load(f)

In [ ]:
json_results['type_d_software']

In [ ]:
with open("hackerNews_updated.json", "r") as f:
    hacker_news=json.load(f)

In [ ]:
hacker_news['0'].keys()

In [ ]:
empty_count=[]
for k,v in hacker_news.items():
    if hacker_news[k]['content'][0]=='Sign up for free and start receiving your daily dose of cybersecurity news, insights and tips.':
        empty_count.append(k)

In [ ]:
len(empty_count)

In [ ]:
empty_count[-100:]

In [ ]:
hacker_news['10002']['content']

In [ ]:
with open("classification_type_results.json", "r") as f:
    json_results=json.load(f)

In [ ]:
json_results.keys()

In [ ]:
json_results['type_a_software'][0:300]

In [ ]:
# CODE FOR CONVERTING SOFTWARE TYPE A TO JSON FORMATS

In [ ]:
import re
from typing import List, Dict, Optional, Tuple


# Matches common version tokens:
# - 2.2.1, 5.1.3.001, 116.0.5845.179
# - 6.7U3, 6.5U3, r43p0, R43p0
VERSION_TOKEN_RE = re.compile(
    r"""
    (?<![A-Za-z0-9])                       # left boundary
    (
        \d+(?:\.\d+)+(?:[A-Za-z][A-Za-z0-9]*)?   # 1.2.3, 6.7U3, 3.1a
        |
        \d+(?:[A-Za-z]\d+)+                     # 6U3 style (rare)
        |
        r\d+p\d+                                 # r43p0 style
    )
    (?![A-Za-z0-9])                         # right boundary
    """,
    re.IGNORECASE | re.VERBOSE
)

# Phrases that imply inequality semantics.
# We'll capture one or multiple versions following these phrases.
RANGE_PATTERNS: List[Tuple[re.Pattern, str]] = [
    # strictly less
    (re.compile(r"\b(prior to|before)\b\s*(" + VERSION_TOKEN_RE.pattern + r")", re.I | re.VERBOSE), "<"),
    # less or equal
    (re.compile(r"\b(up to|upto|through|thru)\b\s*(" + VERSION_TOKEN_RE.pattern + r")", re.I | re.VERBOSE), "<="),
    (re.compile(r"(" + VERSION_TOKEN_RE.pattern + r")\s*\b(and|or)\s*\b(earlier|prior)\b", re.I | re.VERBOSE), "<="),
    (re.compile(r"(" + VERSION_TOKEN_RE.pattern + r")\s*\b(and|or)\s*\b(below|lower)\b", re.I | re.VERBOSE), "<="),
    # strictly greater
    (re.compile(r"\b(after)\b\s*(" + VERSION_TOKEN_RE.pattern + r")", re.I | re.VERBOSE), ">"),
    # greater or equal
    (re.compile(r"(" + VERSION_TOKEN_RE.pattern + r")\s*\b(and|or)\s*\b(later)\b", re.I | re.VERBOSE), ">="),
]

# Extra “Update N” style (kept as a version token if present)
UPDATE_RE = re.compile(r"\bupdate\s*(\d+)\b", re.I)

# Words/phrases to remove from software names (only when they’re version-ish context)
# Keep this aggressive because you explicitly want software names without these tokens.
SOFTWARE_CLEAN_RE = re.compile(
    r"""
    \b(
        versions?|version
        |prior\s+to|before|after
        |up\s+to|upto|through|thru
        |and\s+below|or\s+below|and\s+lower|or\s+lower
        |and\s+earlier|or\s+earlier|and\s+prior|or\s+prior
        |and\s+later|or\s+later
        |earlier|later|prior|below|lower
        |release\s+train
        |update
        |instances?
    )\b
    """,
    re.IGNORECASE | re.VERBOSE
)

TRAILING_JUNK_RE = re.compile(r"[\s,;:\-]+$")


def _dedupe_preserve_order(items: List[str]) -> List[str]:
    seen = set()
    out = []
    for x in items:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out


def extract_semantic_versions(text: str) -> Optional[str]:
    """
    Extract versions with semantics (<, <=, >, >=) based on context words.
    If multiple constraints exist, join with ', ' (deduped).
    Falls back to a single plain version token if no range semantics exist.
    """
    constraints: List[str] = []

    # First, collect inequality constraints
    for pat, op in RANGE_PATTERNS:
        for m in pat.finditer(text):
            # Patterns vary: sometimes version is group 1, sometimes group 2 due to wrapper groups.
            groups = [g for g in m.groups() if g]
            # pick the last group that looks like a version token
            v = None
            for g in reversed(groups):
                if VERSION_TOKEN_RE.fullmatch(g.strip(), re.IGNORECASE):
                    v = g.strip()
                    break
            if v:
                constraints.append(f"{op}{v}")

    # Handle "versions prior to X and Y" / "prior to X and Y" cases:
    # If we saw a keyword like 'prior to'/'through' etc but only captured one,
    # also look for additional version tokens nearby and apply same operator.
    if constraints:
        # Determine operator by first constraint
        op0 = constraints[0][:2] if constraints[0][:2] in ("<=", ">=") else constraints[0][:1]
        # If the text contains "and" and multiple version tokens, add them with the same op
        tokens = VERSION_TOKEN_RE.findall(text)
        if len(tokens) > 1:
            for t in tokens:
                # avoid adding a bare token that already has a constraint
                cand = f"{op0}{t}"
                if cand not in constraints:
                    constraints.append(cand)

        constraints = _dedupe_preserve_order(constraints)
        return ", ".join(constraints)

    # Special case: "Update N and earlier" (or plain "Update N")
    # Only use if it's clearly a version-y phrase
    upd = UPDATE_RE.search(text)
    if upd:
        # If earlier/prior/below context exists, make it <=UpdateN
        if re.search(r"\b(and|or)\s+(earlier|prior|below|lower)\b", text, re.I):
            return f"<=Update {upd.group(1)}"
        return f"Update {upd.group(1)}"

    # Fallback: first plain version token (no semantics)
    m = VERSION_TOKEN_RE.search(text)
    return m.group(1) if m else None


def clean_software_name(software_raw: str) -> str:
    """
    Removes version tokens and version-related words so the software name is clean.
    """
    s = software_raw

    # Remove update phrases like "Update 15"
    s = UPDATE_RE.sub("", s)

    # Remove version-ish words
    s = SOFTWARE_CLEAN_RE.sub("", s)

    # Remove version tokens
    s = VERSION_TOKEN_RE.sub("", s)

    # Collapse leftover punctuation and whitespace
    s = re.sub(r"\(\s*\)", "", s)                 # empty parentheses
    s = re.sub(r"\s{2,}", " ", s).strip()
    s = re.sub(r"\s*,\s*", ", ", s)               # normalize commas
    s = TRAILING_JUNK_RE.sub("", s)

    # Fix occasional dangling "and" from phrases like "versions prior to X and Y"
    s = re.sub(r"\b(and|or)\b$", "", s, flags=re.I).strip()
    s = TRAILING_JUNK_RE.sub("", s)

    return s


def parse_entry(entry: str) -> Dict[str, Optional[str]]:
    """
    Converts 'software, vulnerability' into dict.
    - vulnerability is everything after the first comma (kept as-is)
    - software_version extracted from software text
    """
    parts = [p.strip() for p in entry.split(",", 1)]
    software_raw = parts[0]
    vulnerability = parts[1] if len(parts) > 1 else None

    version = extract_semantic_versions(software_raw)
    software = clean_software_name(software_raw)

    return {
        "software": software if software else software_raw.strip(),
        "software_version": version,
        "vulnerability": vulnerability,
        "url": None,
    }


def convert_data(data: List[List[str]]) -> List[List[Dict[str, Optional[str]]]]:
    """
    Converts list-of-lists input into list-of-lists of dictionaries.
    Keeps empty sublists empty.
    """
    out: List[List[Dict[str, Optional[str]]]] = []
    for sublist in data:
        if not sublist:
            out.append([])
        else:
            out.append([parse_entry(item) for item in sublist])
    return out


In [ ]:
software_type_a_json_converted=convert_data(json_results['type_a_software'])

In [ ]:
software_type_a_json_converted[0:500]

In [ ]:
# CODE FOR CONVERTING HARDWARE TYPE A TO JSON FORMATS

In [ ]:
json_results['type_a_hardware'][0:500]

In [ ]:
import re
from typing import List, Dict, Optional, Tuple


# --- CVE detection (use this to split safely) ---
# Supports:
# - CVE-2023-43261
# - CVE-2023-31168 through CVE-2023-31175
CVE_TAIL_RE = re.compile(
    r"""
    (?P<prefix>.*?)
    (?:,\s*)?
    (?P<vuln>
        CVE-\d{4}-\d{4,7}
        (?:\s+through\s+CVE-\d{4}-\d{4,7})?
    )
    \s*$
    """,
    re.IGNORECASE | re.VERBOSE
)


# --- Version token + patterns (same idea as before, but keep) ---
VERSION_TOKEN_RE = re.compile(
    r"""
    (?<![A-Za-z0-9])
    (
        \d+(?:\.\d+)+(?:[A-Za-z][A-Za-z0-9]*)?
        |
        \d+(?:[A-Za-z]\d+)+
        |
        r\d+p\d+
    )
    (?![A-Za-z0-9])
    """,
    re.IGNORECASE | re.VERBOSE
)

UPDATE_RE = re.compile(r"\bupdate\s*(\d+)\b", re.I)

RANGE_PATTERNS: List[Tuple[re.Pattern, str]] = [
    (re.compile(r"\b(prior to|before)\s+(?:version\s+)?(" + VERSION_TOKEN_RE.pattern + r")", re.I | re.VERBOSE), "<"),
    (re.compile(r"\b(up to|upto|through|thru)\s+(?:version\s+)?(" + VERSION_TOKEN_RE.pattern + r")", re.I | re.VERBOSE), "<="),
    (re.compile(r"(" + VERSION_TOKEN_RE.pattern + r")\s*\b(and|or)\s*\b(earlier|prior)\b", re.I | re.VERBOSE), "<="),
    (re.compile(r"(" + VERSION_TOKEN_RE.pattern + r")\s*\b(and|or)\s*\b(below|lower)\b", re.I | re.VERBOSE), "<="),
    (re.compile(r"\b(after)\s+(?:version\s+)?(" + VERSION_TOKEN_RE.pattern + r")", re.I | re.VERBOSE), ">"),
    (re.compile(r"(" + VERSION_TOKEN_RE.pattern + r")\s*\b(and|or)\s*\b(later)\b", re.I | re.VERBOSE), ">="),
]

TRAILING_JUNK_RE = re.compile(r"[\s,;:\-]+$")

HARDWARE_CLEAN_RE = re.compile(
    r"""
    \b(
        versions?|version
        |prior\s+to|before|after
        |up\s+to|upto|through|thru
        |and\s+below|or\s+below|and\s+lower|or\s+lower
        |and\s+earlier|or\s+earlier|and\s+prior|or\s+prior
        |and\s+later|or\s+later
        |earlier|later|prior|below|lower
        |update
        |devices?|appliances?|routers?|firewalls?|gateways?|switches?|servers?
        |units?|models?|platforms?
    )\b
    """,
    re.IGNORECASE | re.VERBOSE
)


def _dedupe_preserve_order(items: List[str]) -> List[str]:
    seen = set()
    out = []
    for x in items:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out


def extract_semantic_versions(text: str) -> Optional[str]:
    constraints: List[str] = []

    for pat, op in RANGE_PATTERNS:
        for m in pat.finditer(text):
            groups = [g for g in m.groups() if g]
            v = None
            for g in reversed(groups):
                g = g.strip()
                if VERSION_TOKEN_RE.fullmatch(g, re.IGNORECASE):
                    v = g
                    break
            if v:
                constraints.append(f"{op}{v}")

    if constraints:
        op0 = constraints[0][:2] if constraints[0][:2] in ("<=", ">=") else constraints[0][:1]
        tokens = VERSION_TOKEN_RE.findall(text)
        if len(tokens) > 1:
            for t in tokens:
                cand = f"{op0}{t}"
                if cand not in constraints:
                    constraints.append(cand)
        return ", ".join(_dedupe_preserve_order(constraints))

    upd = UPDATE_RE.search(text)
    if upd:
        if re.search(r"\b(and|or)\s+(earlier|prior|below|lower)\b", text, re.I):
            return f"<=Update {upd.group(1)}"
        return f"Update {upd.group(1)}"

    m = VERSION_TOKEN_RE.search(text)
    return m.group(1) if m else None


def clean_hardware_name(hardware_raw: str) -> str:
    s = hardware_raw
    s = UPDATE_RE.sub("", s)
    s = HARDWARE_CLEAN_RE.sub("", s)
    s = VERSION_TOKEN_RE.sub("", s)

    s = re.sub(r"\(\s*\)", "", s)
    s = re.sub(r"\s{2,}", " ", s).strip()
    s = TRAILING_JUNK_RE.sub("", s)

    s = re.sub(r"\b(and|or)\b$", "", s, flags=re.I).strip()
    s = TRAILING_JUNK_RE.sub("", s)
    return s


def split_hardware_and_vuln(entry: str) -> Tuple[str, Optional[str]]:
    """
    Robustly split by trailing CVE instead of first comma.
    If no CVE tail found, fall back to last comma split.
    """
    entry = entry.strip()

    m = CVE_TAIL_RE.match(entry)
    if m:
        hardware_part = m.group("prefix").strip().rstrip(",").strip()
        vuln = m.group("vuln").strip()
        return hardware_part, vuln

    # fallback: last comma
    if "," in entry:
        left, right = entry.rsplit(",", 1)
        return left.strip(), right.strip()

    return entry, None


def parse_hardware_entry(entry: str) -> Dict[str, Optional[str]]:
    hardware_raw, vulnerability = split_hardware_and_vuln(entry)

    version = extract_semantic_versions(hardware_raw)
    hardware = clean_hardware_name(hardware_raw)

    return {
        "hardware": hardware if hardware else hardware_raw.strip(),
        "hardware_version": version,
        "vulnerability": vulnerability,
        "url": None,
    }


def convert_hardware_data(data: List[List[str]]) -> List[List[Dict[str, Optional[str]]]]:
    out: List[List[Dict[str, Optional[str]]]] = []
    for sublist in data:
        if not sublist:
            out.append([])
        else:
            out.append([parse_hardware_entry(item) for item in sublist])
    return out


In [ ]:
hardware_type_a_json_converted=convert_hardware_data(json_results['type_a_hardware'])

In [ ]:
hardware_type_a_json_converted[0:500]

In [ ]:
# CODE FOR CONVERTING SOFTWARE TYPE B TO JSON FORMATS

In [ ]:
json_results['type_b_software'][0:500]

In [ ]:
software_type_b_json_converted=convert_data(json_results['type_b_software'])

In [ ]:
software_type_b_json_converted[0:500]

In [ ]:
# CODE FOR CONVERTING HARDWARE TYPE B TO JSON FORMATS

In [ ]:
json_results['type_b_hardware'][0:500]

In [ ]:
hardware_type_b_json_converted=convert_hardware_data(json_results['type_b_hardware'])

In [ ]:
hardware_type_b_json_converted[0:500]

In [ ]:
# CODE FOR CONVERTING SOFTWARE TYPE C TO JSON FORMATS

In [ ]:
json_results['type_c_software'][0:500]

In [ ]:
import re
from typing import List, Dict, Optional, Tuple, Union, Any


# -----------------------------
# Version extraction (same idea as your improved software parser)
# -----------------------------

VERSION_TOKEN_RE = re.compile(
    r"""
    (?<![A-Za-z0-9])
    (
        \d+(?:\.\d+)+(?:[A-Za-z][A-Za-z0-9]*)?   # 1.2.3, 6.7U3, 3.1a, 9.2.0.006
        |
        \d+(?:[A-Za-z]\d+)+                      # 6U3 style
        |
        r\d+p\d+                                 # r43p0 style
    )
    (?![A-Za-z0-9])
    """,
    re.IGNORECASE | re.VERBOSE
)

UPDATE_RE = re.compile(r"\bupdate\s*(\d+)\b", re.I)

RANGE_PATTERNS: List[Tuple[re.Pattern, str]] = [
    (re.compile(r"\b(prior to|before)\s+(?:version\s+)?(" + VERSION_TOKEN_RE.pattern + r")", re.I | re.VERBOSE), "<"),
    (re.compile(r"\b(up to|upto|through|thru)\s+(?:version\s+)?(" + VERSION_TOKEN_RE.pattern + r")", re.I | re.VERBOSE), "<="),
    (re.compile(r"(" + VERSION_TOKEN_RE.pattern + r")\s*\b(and|or)\s*\b(earlier|prior)\b", re.I | re.VERBOSE), "<="),
    (re.compile(r"(" + VERSION_TOKEN_RE.pattern + r")\s*\b(and|or)\s*\b(below|lower)\b", re.I | re.VERBOSE), "<="),
    (re.compile(r"\b(after)\s+(?:version\s+)?(" + VERSION_TOKEN_RE.pattern + r")", re.I | re.VERBOSE), ">"),
    (re.compile(r"(" + VERSION_TOKEN_RE.pattern + r")\s*\b(and|or)\s*\b(later)\b", re.I | re.VERBOSE), ">="),
]

TRAILING_JUNK_RE = re.compile(r"[\s,;:\-]+$")

SOFTWARE_CLEAN_RE = re.compile(
    r"""
    \b(
        versions?|version|versioning
        |prior\s+to|before|after
        |up\s+to|upto|through|thru
        |and\s+below|or\s+below|and\s+lower|or\s+lower
        |and\s+earlier|or\s+earlier|and\s+prior|or\s+prior
        |and\s+later|or\s+later
        |earlier|later|prior|below|lower
        |release\s+train
        |update
        |instances?
    )\b
    """,
    re.IGNORECASE | re.VERBOSE
)


def _dedupe_preserve_order(items: List[str]) -> List[str]:
    seen = set()
    out = []
    for x in items:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out


def extract_semantic_versions(text: str) -> Optional[str]:
    constraints: List[str] = []

    for pat, op in RANGE_PATTERNS:
        for m in pat.finditer(text):
            groups = [g for g in m.groups() if g]
            v = None
            for g in reversed(groups):
                g = g.strip()
                if VERSION_TOKEN_RE.fullmatch(g, re.IGNORECASE):
                    v = g
                    break
            if v:
                constraints.append(f"{op}{v}")

    if constraints:
        op0 = constraints[0][:2] if constraints[0][:2] in ("<=", ">=") else constraints[0][:1]
        tokens = VERSION_TOKEN_RE.findall(text)
        if len(tokens) > 1:
            for t in tokens:
                cand = f"{op0}{t}"
                if cand not in constraints:
                    constraints.append(cand)
        return ", ".join(_dedupe_preserve_order(constraints))

    upd = UPDATE_RE.search(text)
    if upd:
        if re.search(r"\b(and|or)\s+(earlier|prior|below|lower)\b", text, re.I):
            return f"<=Update {upd.group(1)}"
        return f"Update {upd.group(1)}"

    m = VERSION_TOKEN_RE.search(text)
    return m.group(1) if m else None


def clean_software_name(software_raw: str) -> str:
    s = software_raw
    s = UPDATE_RE.sub("", s)
    s = SOFTWARE_CLEAN_RE.sub("", s)
    s = VERSION_TOKEN_RE.sub("", s)

    s = re.sub(r"\(\s*\)", "", s)  # empty parentheses
    s = re.sub(r"\s{2,}", " ", s).strip()
    s = TRAILING_JUNK_RE.sub("", s)

    s = re.sub(r"\b(and|or)\b$", "", s, flags=re.I).strip()
    s = TRAILING_JUNK_RE.sub("", s)
    return s


# -----------------------------
# New parsing logic for: ["Software, NL mention", "CVE-..."]
# -----------------------------

CVE_RE = re.compile(r"\bCVE-\d{4}-\d{4,7}\b", re.IGNORECASE)


def split_software_and_mention(text: str) -> Tuple[str, Optional[str]]:
    """
    Splits '<software>, <natural language mention>' into (software, mention).
    If no comma exists, mention=None and the whole text is treated as software.
    """
    text = text.strip()
    if "," not in text:
        return text, None

    left, right = text.split(",", 1)
    software_part = left.strip()
    mention_part = right.strip()
    return software_part, (mention_part if mention_part else None)


def parse_software_nl_item(item: Union[List[str], Tuple[str, str]]) -> Dict[str, Optional[str]]:
    """
    Parses one item shaped like:
      ['Confluence, web shell', 'CVE-2023-33533']
    into:
      {
        'software': 'Confluence',
        'software_version': None,
        'vulnerability_mention': 'web shell',
        'vulnerability': 'CVE-2023-33533',
        'url': None
      }
    """
    text, vuln = item[0], item[1]

    # normalize vulnerability (CVE id)
    vuln = vuln.strip() if isinstance(vuln, str) else str(vuln)
    m = CVE_RE.search(vuln)
    vulnerability = m.group(0) if m else vuln  # keep as-is if not standard

    software_raw, mention = split_software_and_mention(str(text))

    # version extraction/cleaning happens on the software chunk only
    software_version = extract_semantic_versions(software_raw)
    software = clean_software_name(software_raw)

    return {
        "software": software if software else software_raw.strip(),
        "software_version": software_version,
        "vulnerability_mention": mention,
        "vulnerability": vulnerability,
        "url": None,
    }


def convert_software_nl_data(data: List[List[Any]]) -> List[List[Dict[str, Optional[str]]]]:
    """
    Converts your list-of-lists into list-of-lists of dicts.
    Keeps empty sublists empty.
    Skips malformed items gracefully (keeps them out rather than crashing).
    """
    out: List[List[Dict[str, Optional[str]]]] = []

    for sublist in data:
        if not sublist:
            out.append([])
            continue

        parsed_sub: List[Dict[str, Optional[str]]] = []
        for item in sublist:
            # Expect list/tuple of length 2
            if isinstance(item, (list, tuple)) and len(item) == 2:
                parsed_sub.append(parse_software_nl_item(item))
            else:
                # If you prefer to keep malformed items as placeholders, replace this `continue`
                # with a dict that stores raw text somewhere.
                continue

        out.append(parsed_sub)

    return out


In [ ]:
software_type_c_json_converted=convert_software_nl_data(json_results['type_c_software'])

In [ ]:
software_type_c_json_converted[0:500]

In [ ]:
json_results['type_c_hardware'][0:500]

In [ ]:
# CODE FOR HARDWARE TYPE C

In [ ]:
import re
from typing import List, Dict, Optional, Tuple, Union, Any


# -----------------------------
# Version extraction (same as before)
# -----------------------------

VERSION_TOKEN_RE = re.compile(
    r"""
    (?<![A-Za-z0-9])
    (
        \d+(?:\.\d+)+(?:[A-Za-z][A-Za-z0-9]*)?   # 1.2.3, 6.7U3, 9.2.0.006
        |
        \d+(?:[A-Za-z]\d+)+                      # 6U3 style
        |
        r\d+p\d+                                 # r43p0 style
    )
    (?![A-Za-z0-9])
    """,
    re.IGNORECASE | re.VERBOSE
)

UPDATE_RE = re.compile(r"\bupdate\s*(\d+)\b", re.I)

RANGE_PATTERNS: List[Tuple[re.Pattern, str]] = [
    (re.compile(r"\b(prior to|before)\s+(?:version\s+)?(" + VERSION_TOKEN_RE.pattern + r")", re.I | re.VERBOSE), "<"),
    (re.compile(r"\b(up to|upto|through|thru)\s+(?:version\s+)?(" + VERSION_TOKEN_RE.pattern + r")", re.I | re.VERBOSE), "<="),
    (re.compile(r"(" + VERSION_TOKEN_RE.pattern + r")\s*\b(and|or)\s*\b(earlier|prior)\b", re.I | re.VERBOSE), "<="),
    (re.compile(r"(" + VERSION_TOKEN_RE.pattern + r")\s*\b(and|or)\s*\b(below|lower)\b", re.I | re.VERBOSE), "<="),
    (re.compile(r"\b(after)\s+(?:version\s+)?(" + VERSION_TOKEN_RE.pattern + r")", re.I | re.VERBOSE), ">"),
    (re.compile(r"(" + VERSION_TOKEN_RE.pattern + r")\s*\b(and|or)\s*\b(later)\b", re.I | re.VERBOSE), ">="),
]

TRAILING_JUNK_RE = re.compile(r"[\s,;:\-]+$")

# Hardware-specific cleaning words (remove these from hardware names only)
HARDWARE_CLEAN_RE = re.compile(
    r"""
    \b(
        versions?|version|versioning
        |prior\s+to|before|after
        |up\s+to|upto|through|thru
        |and\s+below|or\s+below|and\s+lower|or\s+lower
        |and\s+earlier|or\s+earlier|and\s+prior|or\s+prior
        |and\s+later|or\s+later
        |earlier|later|prior|below|lower
        |update
        |devices?|appliances?|routers?|firewalls?|gateways?|switches?|servers?
        |units?|models?|platforms?|instances?
    )\b
    """,
    re.IGNORECASE | re.VERBOSE
)

CVE_RE = re.compile(r"\bCVE-\d{4}-\d{4,7}\b", re.IGNORECASE)


def _dedupe_preserve_order(items: List[str]) -> List[str]:
    seen = set()
    out = []
    for x in items:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out


def extract_semantic_versions(text: str) -> Optional[str]:
    constraints: List[str] = []

    for pat, op in RANGE_PATTERNS:
        for m in pat.finditer(text):
            groups = [g for g in m.groups() if g]
            v = None
            for g in reversed(groups):
                g = g.strip()
                if VERSION_TOKEN_RE.fullmatch(g, re.IGNORECASE):
                    v = g
                    break
            if v:
                constraints.append(f"{op}{v}")

    if constraints:
        op0 = constraints[0][:2] if constraints[0][:2] in ("<=", ">=") else constraints[0][:1]
        tokens = VERSION_TOKEN_RE.findall(text)
        if len(tokens) > 1:
            for t in tokens:
                cand = f"{op0}{t}"
                if cand not in constraints:
                    constraints.append(cand)
        return ", ".join(_dedupe_preserve_order(constraints))

    upd = UPDATE_RE.search(text)
    if upd:
        if re.search(r"\b(and|or)\s+(earlier|prior|below|lower)\b", text, re.I):
            return f"<=Update {upd.group(1)}"
        return f"Update {upd.group(1)}"

    m = VERSION_TOKEN_RE.search(text)
    return m.group(1) if m else None


def clean_hardware_name(hardware_raw: str) -> str:
    s = hardware_raw
    s = UPDATE_RE.sub("", s)
    s = HARDWARE_CLEAN_RE.sub("", s)
    s = VERSION_TOKEN_RE.sub("", s)

    s = re.sub(r"\(\s*\)", "", s)     # empty parentheses
    s = re.sub(r"\s{2,}", " ", s).strip()
    s = TRAILING_JUNK_RE.sub("", s)

    s = re.sub(r"\b(and|or)\b$", "", s, flags=re.I).strip()
    s = TRAILING_JUNK_RE.sub("", s)
    return s


def split_hardware_and_mention(text: str) -> Tuple[str, Optional[str]]:
    """
    Splits '<hardware>, <natural language mention>' into (hardware, mention).
    If no comma exists, mention=None and the whole text is treated as hardware.
    """
    text = text.strip()
    if "," not in text:
        return text, None
    left, right = text.split(",", 1)
    hardware_part = left.strip()
    mention_part = right.strip()
    return hardware_part, (mention_part if mention_part else None)


def parse_hardware_nl_item(item: Union[List[str], Tuple[str, str]]) -> Dict[str, Optional[str]]:
    """
    Parses:
      ['NetScaler ADC, session hijacking', 'CVE-2016-6845']
    into:
      {
        'hardware': 'NetScaler ADC',
        'hardware_version': None,
        'vulnerability_mention': 'session hijacking',
        'vulnerability': 'CVE-2016-6845',
        'url': None
      }
    """
    text, vuln = item[0], item[1]

    vuln = vuln.strip() if isinstance(vuln, str) else str(vuln)
    m = CVE_RE.search(vuln)
    vulnerability = m.group(0) if m else vuln

    hardware_raw, mention = split_hardware_and_mention(str(text))

    hardware_version = extract_semantic_versions(hardware_raw)
    hardware = clean_hardware_name(hardware_raw)

    return {
        "hardware": hardware if hardware else hardware_raw.strip(),
        "hardware_version": hardware_version,
        "vulnerability_mention": mention,
        "vulnerability": vulnerability,
        "url": None,
    }


def convert_hardware_nl_data(data: List[List[Any]]) -> List[List[Dict[str, Optional[str]]]]:
    """
    Converts list-of-lists input into list-of-lists of hardware dicts.
    Keeps empty sublists empty.
    Skips malformed items (non [text, cve]) to avoid crashing.
    """
    out: List[List[Dict[str, Optional[str]]]] = []

    for sublist in data:
        if not sublist:
            out.append([])
            continue

        parsed_sub: List[Dict[str, Optional[str]]] = []
        for item in sublist:
            if isinstance(item, (list, tuple)) and len(item) == 2:
                parsed_sub.append(parse_hardware_nl_item(item))
            else:
                continue

        out.append(parsed_sub)

    return out


In [ ]:
hardware_type_c_json_converted=convert_hardware_nl_data(json_results['type_c_hardware'])

In [ ]:
hardware_type_c_json_converted[0:500]

In [ ]:
#CODE FOR TYPE D SOFTWARE

In [ ]:
json_results['type_d_software'][0:500]

In [ ]:
import re
from typing import List, Dict, Optional, Tuple, Union, Any


# -----------------------------
# Version extraction (same as before)
# -----------------------------

VERSION_TOKEN_RE = re.compile(
    r"""
    (?<![A-Za-z0-9])
    (
        \d+(?:\.\d+)+(?:[A-Za-z][A-Za-z0-9]*)?
        |
        \d+(?:[A-Za-z]\d+)+
        |
        r\d+p\d+
    )
    (?![A-Za-z0-9])
    """,
    re.IGNORECASE | re.VERBOSE
)

UPDATE_RE = re.compile(r"\bupdate\s*(\d+)\b", re.I)

RANGE_PATTERNS = [
    (re.compile(r"\b(prior to|before)\s+(?:version\s+)?(" + VERSION_TOKEN_RE.pattern + r")", re.I | re.VERBOSE), "<"),
    (re.compile(r"\b(up to|upto|through|thru)\s+(?:version\s+)?(" + VERSION_TOKEN_RE.pattern + r")", re.I | re.VERBOSE), "<="),
    (re.compile(r"(" + VERSION_TOKEN_RE.pattern + r")\s*\b(and|or)\s*\b(earlier|prior)\b", re.I | re.VERBOSE), "<="),
    (re.compile(r"(" + VERSION_TOKEN_RE.pattern + r")\s*\b(and|or)\s*\b(below|lower)\b", re.I | re.VERBOSE), "<="),
    (re.compile(r"\b(after)\s+(?:version\s+)?(" + VERSION_TOKEN_RE.pattern + r")", re.I | re.VERBOSE), ">"),
    (re.compile(r"(" + VERSION_TOKEN_RE.pattern + r")\s*\b(and|or)\s*\b(later)\b", re.I | re.VERBOSE), ">="),
]

TRAILING_JUNK_RE = re.compile(r"[\s,;:\-]+$")

SOFTWARE_CLEAN_RE = re.compile(
    r"""
    \b(
        versions?|version|versioning
        |prior\s+to|before|after
        |up\s+to|upto|through|thru
        |and\s+below|or\s+below|and\s+lower|or\s+lower
        |and\s+earlier|or\s+earlier|and\s+prior|or\s+prior
        |and\s+later|or\s+later
        |earlier|later|prior|below|lower
        |release\s+train
        |update
        |instances?
    )\b
    """,
    re.IGNORECASE | re.VERBOSE
)


def _dedupe_preserve_order(items: List[str]) -> List[str]:
    seen = set()
    out = []
    for x in items:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out


def extract_semantic_versions(text: str) -> Optional[str]:
    constraints: List[str] = []

    for pat, op in RANGE_PATTERNS:
        for m in pat.finditer(text):
            groups = [g for g in m.groups() if g]
            v = None
            for g in reversed(groups):
                g = g.strip()
                if VERSION_TOKEN_RE.fullmatch(g, re.IGNORECASE):
                    v = g
                    break
            if v:
                constraints.append(f"{op}{v}")

    if constraints:
        op0 = constraints[0][:2] if constraints[0][:2] in ("<=", ">=") else constraints[0][:1]
        tokens = VERSION_TOKEN_RE.findall(text)
        if len(tokens) > 1:
            for t in tokens:
                cand = f"{op0}{t}"
                if cand not in constraints:
                    constraints.append(cand)
        return ", ".join(_dedupe_preserve_order(constraints))

    upd = UPDATE_RE.search(text)
    if upd:
        if re.search(r"\b(and|or)\s+(earlier|prior|below|lower)\b", text, re.I):
            return f"<=Update {upd.group(1)}"
        return f"Update {upd.group(1)}"

    m = VERSION_TOKEN_RE.search(text)
    return m.group(1) if m else None


def clean_software_name(software_raw: str) -> str:
    s = software_raw
    s = UPDATE_RE.sub("", s)
    s = SOFTWARE_CLEAN_RE.sub("", s)
    s = VERSION_TOKEN_RE.sub("", s)

    s = re.sub(r"\(\s*\)", "", s)
    s = re.sub(r"\s{2,}", " ", s).strip()
    s = TRAILING_JUNK_RE.sub("", s)

    s = re.sub(r"\b(and|or)\b$", "", s, flags=re.I).strip()
    s = TRAILING_JUNK_RE.sub("", s)
    return s


# -----------------------------
# CWE parsing
# -----------------------------

CWE_RE = re.compile(r"\bCWE-\d+\b", re.IGNORECASE)


def split_software_and_mention(text: str) -> Tuple[str, Optional[str]]:
    """
    Splits '<software>, <natural language mention>' into (software, mention).
    If no comma, mention=None.
    """
    text = str(text).strip()
    if "," not in text:
        return text, None
    left, right = text.split(",", 1)
    software_part = left.strip()
    mention_part = right.strip()
    return software_part, (mention_part if mention_part else None)


def parse_software_cwe_item(item: Union[List[str], Tuple[str, str]]) -> Dict[str, Optional[str]]:
    """
    Parses:
      ['ChatGPT, SQL injection', 'CWE-89']
    into dict with:
      software, software_version, vulnerability_mention, vulnerability(=CWE-*), url
    """
    text, cwe = item[0], item[1]

    cwe = cwe.strip() if isinstance(cwe, str) else str(cwe)
    m = CWE_RE.search(cwe)
    vulnerability = m.group(0) if m else cwe

    software_raw, mention = split_software_and_mention(text)

    software_version = extract_semantic_versions(software_raw)
    software = clean_software_name(software_raw)

    return {
        "software": software if software else str(software_raw).strip(),
        "software_version": software_version,
        "vulnerability_mention": mention,
        "vulnerability": vulnerability,
        "url": None,
    }


def convert_software_cwe_data(data: List[List[Any]]) -> List[List[Dict[str, Optional[str]]]]:
    """
    Converts list-of-lists into list-of-lists of dicts.
    Keeps empty sublists empty.
    Skips malformed items (not length-2 sequences).
    """
    out: List[List[Dict[str, Optional[str]]]] = []

    for sublist in data:
        if not sublist:
            out.append([])
            continue

        parsed: List[Dict[str, Optional[str]]] = []
        for item in sublist:
            if isinstance(item, (list, tuple)) and len(item) == 2:
                parsed.append(parse_software_cwe_item(item))
            else:
                continue

        out.append(parsed)

    return out


In [ ]:
software_type_d_json_converted=convert_software_cwe_data(json_results['type_d_software'])

In [ ]:
software_type_d_json_converted[0:500]

In [ ]:
#CODE FOR TYPE D HARDWARE

In [ ]:
json_results['type_d_hardware'][0:1000]

In [ ]:
import re
from typing import List, Dict, Optional, Tuple, Union, Any


# -----------------------------
# Version extraction (same as before)
# -----------------------------

VERSION_TOKEN_RE = re.compile(
    r"""
    (?<![A-Za-z0-9])
    (
        \d+(?:\.\d+)+(?:[A-Za-z][A-Za-z0-9]*)?   # 1.2.3, 6.7U3, 9.2.0.006
        |
        \d+(?:[A-Za-z]\d+)+                      # 6U3 style
        |
        r\d+p\d+                                 # r43p0 style
    )
    (?![A-Za-z0-9])
    """,
    re.IGNORECASE | re.VERBOSE
)

UPDATE_RE = re.compile(r"\bupdate\s*(\d+)\b", re.I)

RANGE_PATTERNS: List[Tuple[re.Pattern, str]] = [
    (re.compile(r"\b(prior to|before)\s+(?:version\s+)?(" + VERSION_TOKEN_RE.pattern + r")", re.I | re.VERBOSE), "<"),
    (re.compile(r"\b(up to|upto|through|thru)\s+(?:version\s+)?(" + VERSION_TOKEN_RE.pattern + r")", re.I | re.VERBOSE), "<="),
    (re.compile(r"(" + VERSION_TOKEN_RE.pattern + r")\s*\b(and|or)\s*\b(earlier|prior)\b", re.I | re.VERBOSE), "<="),
    (re.compile(r"(" + VERSION_TOKEN_RE.pattern + r")\s*\b(and|or)\s*\b(below|lower)\b", re.I | re.VERBOSE), "<="),
    (re.compile(r"\b(after)\s+(?:version\s+)?(" + VERSION_TOKEN_RE.pattern + r")", re.I | re.VERBOSE), ">"),
    (re.compile(r"(" + VERSION_TOKEN_RE.pattern + r")\s*\b(and|or)\s*\b(later)\b", re.I | re.VERBOSE), ">="),
]

TRAILING_JUNK_RE = re.compile(r"[\s,;:\-]+$")

HARDWARE_CLEAN_RE = re.compile(
    r"""
    \b(
        versions?|version|versioning
        |prior\s+to|before|after
        |up\s+to|upto|through|thru
        |and\s+below|or\s+below|and\s+lower|or\s+lower
        |and\s+earlier|or\s+earlier|and\s+prior|or\s+prior
        |and\s+later|or\s+later
        |earlier|later|prior|below|lower
        |release\s+train
        |update
        |instances?
        |devices?|appliances?|routers?|firewalls?|gateways?|switches?|servers?
        |units?|models?|platforms?
    )\b
    """,
    re.IGNORECASE | re.VERBOSE
)


def _dedupe_preserve_order(items: List[str]) -> List[str]:
    seen = set()
    out = []
    for x in items:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out


def extract_semantic_versions(text: str) -> Optional[str]:
    constraints: List[str] = []

    for pat, op in RANGE_PATTERNS:
        for m in pat.finditer(text):
            groups = [g for g in m.groups() if g]
            v = None
            for g in reversed(groups):
                g = g.strip()
                if VERSION_TOKEN_RE.fullmatch(g, re.IGNORECASE):
                    v = g
                    break
            if v:
                constraints.append(f"{op}{v}")

    if constraints:
        op0 = constraints[0][:2] if constraints[0][:2] in ("<=", ">=") else constraints[0][:1]
        tokens = VERSION_TOKEN_RE.findall(text)
        if len(tokens) > 1:
            for t in tokens:
                cand = f"{op0}{t}"
                if cand not in constraints:
                    constraints.append(cand)
        return ", ".join(_dedupe_preserve_order(constraints))

    upd = UPDATE_RE.search(text)
    if upd:
        if re.search(r"\b(and|or)\s+(earlier|prior|below|lower)\b", text, re.I):
            return f"<=Update {upd.group(1)}"
        return f"Update {upd.group(1)}"

    m = VERSION_TOKEN_RE.search(text)
    return m.group(1) if m else None


def clean_hardware_name(hardware_raw: str) -> str:
    s = str(hardware_raw).strip()
    s = UPDATE_RE.sub("", s)
    s = HARDWARE_CLEAN_RE.sub("", s)
    s = VERSION_TOKEN_RE.sub("", s)

    s = re.sub(r"\(\s*\)", "", s)
    s = re.sub(r"\s{2,}", " ", s).strip()
    s = TRAILING_JUNK_RE.sub("", s)

    s = re.sub(r"\b(and|or)\b$", "", s, flags=re.I).strip()
    s = TRAILING_JUNK_RE.sub("", s)
    return s


# -----------------------------
# CWE parsing + splitting "hardware, mention"
# -----------------------------

CWE_RE = re.compile(r"\bCWE-\d+\b", re.IGNORECASE)


def split_hardware_and_mention(text: str) -> Tuple[str, Optional[str]]:
    """
    Splits '<hardware>, <natural language mention>' into (hardware, mention).
    If no comma, mention=None.
    """
    text = str(text).strip()
    if "," not in text:
        return text, None
    left, right = text.split(",", 1)
    hardware_part = left.strip()
    mention_part = right.strip()
    return hardware_part, (mention_part if mention_part else None)


def parse_hardware_cwe_item(item: Union[List[str], Tuple[str, str]]) -> Dict[str, Optional[str]]:
    """
    Parses:
      ['NetScaler ADC, session hijacking', 'CWE-601']
    into dict with:
      hardware, hardware_version, vulnerability_mention, vulnerability(=CWE-*), url
    """
    text, cwe = item[0], item[1]

    cwe = cwe.strip() if isinstance(cwe, str) else str(cwe)
    m = CWE_RE.search(cwe)
    vulnerability = m.group(0) if m else cwe

    hardware_raw, mention = split_hardware_and_mention(text)

    hardware_version = extract_semantic_versions(hardware_raw)
    hardware = clean_hardware_name(hardware_raw)

    return {
        "hardware": hardware if hardware else str(hardware_raw).strip(),
        "hardware_version": hardware_version,
        "vulnerability_mention": mention,
        "vulnerability": vulnerability,
        "url": None,
    }


def convert_hardware_cwe_data(data: List[List[Any]]) -> List[List[Dict[str, Optional[str]]]]:
    """
    Converts list-of-lists into list-of-lists of dicts.
    Keeps empty sublists empty.
    Skips malformed items (not length-2 sequences).
    """
    out: List[List[Dict[str, Optional[str]]]] = []

    for sublist in data:
        if not sublist:
            out.append([])
            continue

        parsed: List[Dict[str, Optional[str]]] = []
        for item in sublist:
            if isinstance(item, (list, tuple)) and len(item) == 2:
                parsed.append(parse_hardware_cwe_item(item))
            else:
                continue

        out.append(parsed)

    return out


In [ ]:
hardware_type_d_json_converted=convert_hardware_cwe_data(json_results['type_d_hardware'])

In [ ]:
hardware_type_d_json_converted[0:1000]

In [ ]:
#CODE FOR SOFTWARE TYPE E

In [ ]:
json_results['type_e_software'][0:500]

In [ ]:
from typing import List, Dict, Optional


def parse_software_nl_item(text: str) -> Dict[str, Optional[str]]:
    """
    Parses:
      'Software, natural language vulnerability'
    into:
      {software, vulnerability_mention, url}
    """
    text = text.strip()

    if "," in text:
        software, mention = text.split(",", 1)
        software = software.strip()
        mention = mention.strip()
    else:
        software = text
        mention = None

    return {
        "software": software,
        "vulnerability_mention": mention,
        "url": None
    }


def convert_software_nl_data(data: List[List[str]]) -> List[List[Dict[str, Optional[str]]]]:
    """
    Converts list-of-lists of strings into structured dictionaries.
    Keeps empty sublists empty.
    """
    output: List[List[Dict[str, Optional[str]]]] = []

    for sublist in data:
        if not sublist:
            output.append([])
        else:
            output.append([parse_software_nl_item(item) for item in sublist])

    return output


In [ ]:
software_type_e_json_converted=convert_software_nl_data(json_results['type_e_software'])

In [ ]:
software_type_e_json_converted[0:1000]

In [ ]:
#CODE FOR HARDWARE

In [ ]:
json_results['type_e_hardware'][0:500]

In [ ]:
from typing import List, Dict, Optional


def parse_hardware_nl_item(text: str) -> Dict[str, Optional[str]]:
    """
    Parses:
      'Hardware, natural language vulnerability'
    into:
      {hardware, vulnerability_mention, url}
    """
    text = text.strip()

    if "," in text:
        hardware, mention = text.split(",", 1)
        hardware = hardware.strip()
        mention = mention.strip()
    else:
        hardware = text
        mention = None

    return {
        "hardware": hardware,
        "vulnerability_mention": mention,
        "url": None
    }


def convert_hardware_nl_data(data: List[List[str]]) -> List[List[Dict[str, Optional[str]]]]:
    """
    Converts list-of-lists of strings into structured hardware dictionaries.
    Keeps empty sublists empty.
    """
    output: List[List[Dict[str, Optional[str]]]] = []

    for sublist in data:
        if not sublist:
            output.append([])
        else:
            output.append([parse_hardware_nl_item(item) for item in sublist])

    return output


In [ ]:
hardware_type_e_json_converted=convert_hardware_nl_data(json_results['type_e_hardware'])

In [ ]:
hardware_type_e_json_converted[0:100]

In [ ]:
combined={
    'software_type_a': software_type_a_json_converted,
    'software_type_b': software_type_b_json_converted,
    'software_type_c': software_type_c_json_converted,
    'software_type_d': software_type_d_json_converted,
    'software_type_e': software_type_e_json_converted,
    'hardware_type_a': hardware_type_a_json_converted,
    'hardware_type_b': hardware_type_b_json_converted,
    'hardware_type_c': hardware_type_c_json_converted,
    'hardware_type_d': hardware_type_d_json_converted,
    'hardware_type_e': hardware_type_e_json_converted
}

In [ ]:
for k,v in combined.items():
    count=0
    print(k)
    for i in range(len(v)):
        if len(v[i]) > 0:
            count+=len(v[i])
    print(count)

In [ ]:
with open("classification_types_in_json.json", "w") as f:
    json.dump(combined, f, indent=2)